# 01 VAE Basics

这个 notebook 用一个小型 MLP VAE 帮助理解作业中的基础内容。目标不是先追求漂亮图片，而是看清楚 encoder、潜变量、重参数化、decoder、重构损失和 KL 正则化之间的关系。


## VAE 要解决什么问题

普通 autoencoder 学到的是把输入压缩再重构。VAE 额外要求潜变量来自一个可采样的概率空间。这样训练结束后，我们可以从先验分布 `N(0, I)` 采样 latent vector，再用 decoder 生成新图像。


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from vae_project.config import validate_config
from vae_project.data import get_dataloaders
from vae_project.models import MLPVAE
from vae_project.losses import vae_loss
from vae_project.train import train_one_epoch, evaluate_epoch
from vae_project.utils import select_device, set_seed
from vae_project.visualization import save_prior_samples, save_reconstruction_grid


## 数据设置

默认的 `dataset: 'fake'` 使用 FakeData；`fake`（FakeData）只用于离线 smoke/学习，方便在没有下载数据时跑通 notebook。需要在真实数据上调试代码时，把 `dataset` 改成 `mnist`；`mnist`（MNIST）用于真实数据上的代码调试。作业的主实验/正式实验必须使用 `fashion_mnist`；`fashion_mnist`（Fashion-MNIST）是作业要求的主实验/正式实验数据集，并应在此数据集上比较不同 beta。
图像保持在 `[0, 1]`，所以可以使用 Bernoulli/BCE 风格的重构损失。


In [ ]:
config = validate_config({
    'run_name': 'notebook_smoke',
    'dataset': 'fake',
    'data_dir': 'data',
    'output_dir': 'outputs/notebook_smoke',
    'batch_size': 32,
    'epochs': 1,
    'learning_rate': 1e-3,
    'seed': 42,
    'device': 'auto',
    'beta': 1.0,
    'latent_dim': 8,
    'hidden_dims': [128, 64],
    'train_limit': 128,
    'test_limit': 64,
    'num_workers': 0,
    'download': False,
    'sample_count': 16,
})
set_seed(config['seed'])
device = select_device(config['device'])
train_loader, test_loader = get_dataloaders(config)
device


## Encoder 输出 mu 和 logvar

Encoder 不直接输出一个固定 latent vector，而是输出近似后验 `q_phi(z|x)` 的均值 `mu` 和对数方差 `logvar`。这样每张图像对应的是一个分布，而不是单个点。


In [ ]:
model = MLPVAE(input_shape=(1, 28, 28), hidden_dims=config['hidden_dims'], latent_dim=config['latent_dim']).to(device)
images, labels = next(iter(train_loader))
images = images.to(device)
mu, logvar = model.encode(images)
mu.shape, logvar.shape


## 重参数化技巧

直接从 `N(mu, sigma^2)` 采样会阻断梯度路径。VAE 写成 `z = mu + std * eps`，其中 `eps ~ N(0, I)`。随机性来自 `eps`，梯度可以通过 `mu` 和 `std` 回传。


In [ ]:
model.train()
z = model.reparameterize(mu, logvar)
recon_logits = model.decode(z)
z.shape, recon_logits.shape


## ELBO 损失

作业目标可以写成重构项加 `beta * KL(q(z|x) || p(z))`。重构项鼓励 decoder 还原输入；KL 项鼓励近似后验接近标准正态先验，使先验采样更有意义。


In [ ]:
output = model(images)
losses = vae_loss(output['recon_logits'], images, output['mu'], output['logvar'], beta=config['beta'])
{name: float(value.detach().cpu()) for name, value in losses.items()}


## 训练一个小型 VAE

这里先跑一个很小的训练流程，确认代码路径正确。正式实验会用 CLI 在 Fashion-MNIST 上跑 `beta=1` 和 `beta=0`。


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
history = []
for epoch in range(1, config['epochs'] + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device, beta=config['beta'])
    test_metrics = evaluate_epoch(model, test_loader, device, beta=config['beta'])
    row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'test_{k}': v for k, v in test_metrics.items()}}
    history.append(row)
history


## 重构测试图像

上面一行是原图，下面一行是重构图。刚开始训练时重构会比较模糊，这是正常现象。


In [ ]:
figures_dir = Path(config['output_dir']) / 'figures'
save_reconstruction_grid(model, test_loader, device, figures_dir / 'notebook_reconstructions.png', max_images=8)
plt.imshow(plt.imread(figures_dir / 'notebook_reconstructions.png'))
plt.axis('off');


## 从先验分布采样

训练好的 VAE 可以从 `N(0, I)` 采样 latent vector，然后用 decoder 生成图像。`beta=0` 时重构可能变好，但先验采样通常会变差，因为 encoder 学到的 latent 分布不再被约束靠近先验。


In [ ]:
save_prior_samples(model, device, figures_dir / 'notebook_prior_samples.png', sample_count=16)
plt.imshow(plt.imread(figures_dir / 'notebook_prior_samples.png'))
plt.axis('off');


## beta=1 和 beta=0

`beta=1` 是标准 VAE。`beta=0` 移除 KL 正则化，模型更像带随机采样的 autoencoder。基础实验会固定网络结构和训练设置，只改变 beta，然后比较重构、KL、先验采样和损失曲线。
